In [1]:
using Pkg

Pkg.activate("../../../../")
using RigidBodyDynamics
using MeshCatMechanisms
using MeshCat

  Activating project at `~/planning_on_biped_robot/Code/MPCBipedRobot`


# Visualiser
The purpose of this notebook is to visualise the built urdfs with onshape, and identify potential issues.
There is a separate environment for this notebook because there are compatibility issues between MeshCatMechanisms and ModelingToolkit for instance.

## 1. Load mechanism

In [2]:
urdf_path = "robot.urdf"
robot = parse_urdf(Float64, urdf_path)
remove_fixed_tree_joints!(robot)
joints(robot)

6-element Vector{Joint{Float64, JT} where JT<:JointType{Float64}}:
 Joint "hip_left": Revolute joint with axis [0.0, 0.0, 1.0]
 Joint "hip_right": Revolute joint with axis [0.0, 0.0, 1.0]
 Joint "knee_left": Revolute joint with axis [0.0, 0.0, 1.0]
 Joint "knee_right": Revolute joint with axis [0.0, 0.0, 1.0]
 Joint "ankle_left": Revolute joint with axis [0.0, 0.0, 1.0]
 Joint "ankle_right": Revolute joint with axis [0.0, 0.0, 1.0]

## 2. Visualisation

In [3]:
vis = MechanismVisualizer(robot, URDFVisuals(urdf_path));

┌ Info: Listening on: 127.0.0.1:8711, thread id: 1
└ @ HTTP.Servers /home/brix/.julia/packages/HTTP/4AUPl/src/Servers.jl:382
┌ Info: MeshCat server started. You can open the visualizer by visiting the following URL in your browser:
│ http://127.0.0.1:8711
└ @ MeshCat /home/brix/.julia/packages/MeshCat/9QrxD/src/visualizer.jl:43


## 3. Further analysis

### Define and set the state

In [4]:
state = MechanismState(robot)

MechanismState{Float64, Float64, Float64, …}(…)

In [5]:
hip_left, hip_right,  knee_left, knee_right, ankle_left, ankle_right = joints(robot)


6-element Vector{Joint{Float64, JT} where JT<:JointType{Float64}}:
 Joint "hip_left": Revolute joint with axis [0.0, 0.0, 1.0]
 Joint "hip_right": Revolute joint with axis [0.0, 0.0, 1.0]
 Joint "knee_left": Revolute joint with axis [0.0, 0.0, 1.0]
 Joint "knee_right": Revolute joint with axis [0.0, 0.0, 1.0]
 Joint "ankle_left": Revolute joint with axis [0.0, 0.0, 1.0]
 Joint "ankle_right": Revolute joint with axis [0.0, 0.0, 1.0]

In [6]:
set_configuration!(state, hip_right, pi/2)
set_configuration!(state, knee_right, pi/4)
set_configuration!(state, ankle_right, pi/3)
set_configuration!(state, hip_left, pi/2)
set_configuration!(state, knee_left, pi/4)
set_configuration!(state, ankle_left, pi/3)

# set_configuration!(state, hip_right, pi/10)
# set_configuration!(state, knee_right, pi/10)
# set_configuration!(state, hip_left, pi/10)
# set_configuration!(state, knee_left, pi/10)

# set_configuration!(state, hip_right, 0)
# set_configuration!(state, knee_right, 0)
# set_configuration!(state, hip_left, 0)
# set_configuration!(state, knee_left, 0)

# set_configuration!(state, foot, 0)

# set_velocity!(state, hip_right, 10)
# set_velocity!(state, knee_right, 0)
# set_velocity!(state, hip_left, 0)
# set_velocity!(state, knee_left, 0)
zero_velocity!(state)
# **Important**: a `MechanismState` contains cache variables that depend on the configurations and velocities of the joints. These need to be invalidated when the configurations and velocities are changed. To do this, call
setdirty!(state)

q = configuration(state)
v = velocity(state)
print("q=$q\nv=$v")

q=[1.5707963267948966, 1.5707963267948966, 0.7853981633974483, 0.7853981633974483, 1.0471975511965976, 1.0471975511965976]
v=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

How much does the robot weigth and where is its com ? 


In [4]:
println("weight: ", mass(robot), "kg")
println(center_of_mass(state))

println(bodies(robot))
for body in bodies(robot)[2:end]
    inertia = spatial_inertia(body)
    mass = inertia.mass
    moment = inertia.moment
    println("Body: $(body.name), Mass: $mass kg, Moment Inertia: $moment ")
end 

weight: 0.7464674kg
Point3D in "world": [1.2322924792401622e-7, 0.04999926619953373, -0.2401082916733701]
RigidBody{Float64}[RigidBody: "world", RigidBody: "leg_top_full_asmb", RigidBody: "leg_top_full_asmb_2", RigidBody: "leg_bottom_asmb", RigidBody: "leg_bottom_asmb_2", RigidBody: "feet_asmb", RigidBody: "feet_asmb_2"]
Body: leg_top_full_asmb, Mass: 0.239756 kg, Moment Inertia: [0.005836551331243948 2.7699132710039325e-7 2.9092958180847117e-8; 2.7699132710039325e-7 0.00012580724362753516 -0.0006548647004945759; 2.909295818084712e-8 -0.0006548647004945759 0.005744076306658275] 
Body: leg_top_full_asmb_2, Mass: 0.239756 kg, Moment Inertia: [0.005855920738971948 -2.7699132710039325e-7 -3.211467977076712e-8; -2.7699132710039325e-7 0.00014517665135553495 -0.0007227962065105759; -3.2114679770767127e-8 -0.0007227962065105759 0.005744076306658274] 
Body: leg_bottom_asmb, Mass: 0.112193 kg, Moment Inertia: [0.0012481245334887738 1.8090161162264242e-8 1.0260048285422023e-8; 1.809016116226425e-

Now let's compute the endEffector position


In [ ]:
endEffector = ("feet_asmb" , "feet_asmb_2")
for effector in endEffector
    foot_link = findbody(robot, "$effector")
    frame = default_frame(foot_link)
    tf_world_to_body = transform_to_root(state, frame)
    println(tf_world_to_body)
end 

hip_boom = findbody(robot, "hip_boom")
frame = default_frame(hip_boom)
tf_world_to_body = transform_to_root(state, frame)
println(tf_world_to_body)

### Simulation

In [7]:
ts, qs, vs = simulate(state, 1., Δt = 1e-3);

In [8]:
mvis = MechanismVisualizer(robot, URDFVisuals(urdf_path))
animation = Animation(mvis, ts, qs)
setanimation!(mvis, animation)

# Create a MechanismVisualizer and visualize
MeshCatMechanisms.animate(mvis, ts, qs; realtimerate = 1.)

┌ Info: Listening on: 127.0.0.1:8712, thread id: 1
└ @ HTTP.Servers /home/brix/.julia/packages/HTTP/4AUPl/src/Servers.jl:382
┌ Info: MeshCat server started. You can open the visualizer by visiting the following URL in your browser:
│ http://127.0.0.1:8712
└ @ MeshCat /home/brix/.julia/packages/MeshCat/9QrxD/src/visualizer.jl:43


## Same with model_urdf.jl

In [ ]:
include("model_urdf.jl")
get_mechanism(wanted_mech ="double_pendulum")